# Tien herhaalde opwarmmetingen: supergeleider, offsetcompensatie en B–T

Deze notebook verwerkt **10 doorlopende runs**. Elke run bestaat uit één DMM-bestand (PT1000-weerstand) en één SMU-bestand (spanning). Per run verandert de stroom in deze volgorde:

`+0.1 → −0.1 → +0.3 → −0.3 → +0.5 → −0.5 → +0.7 → −0.7 A`

De analyse:
1. leest de ene numerieke kolom uit elk bestand;
2. rekent PT1000-weerstand om naar temperatuur met jullie kalibratie;
3. deelt iedere run op in stroomblokken;
4. zoekt positieve en negatieve stroommetingen **bij dezelfde temperatuur** op temperatuur-bin niveau;
5. berekent offsetgecorrigeerde weerstandscurves en spreiding over runs;
6. schat eventueel per stroom de overgangstemperatuur en maakt een verkennende B–T-plot.

**Belangrijke beperking van de exports:** de getoonde bestanden hebben geen tijdstempels. De code koppelt DMM en SMU daarom op meetindex. Dat is alleen verantwoord als beide instrumenten synchroon gestart zijn en dezelfde samplecadans hebben. Ook moeten de meetindexen van de stroomwisselingen kloppen.

De offsetcompensatie gebruikt de practicumformule
\[
V_{\mathrm{sample}}(T)=\frac{V(+I,T)-V(-I,T)}{2},\qquad R(T)=\frac{V(+I,T)-V(-I,T)}{2I}.
\]
De positieve en negatieve metingen worden niet blind op relatieve voortgang gekoppeld, maar alleen in dezelfde temperatuur-bin **binnen dezelfde run**. Zo voorkomen we dat een temperatuurstijging tussen stroomblokken ongemerkt als offset wordt behandeld. Als er weinig overlap is, zal de notebook dat melden in plaats van data te extrapoleren.

## 1. Instellingen invullen

Pas vooral de bestandsnaam-template aan de echte SMU-bestandsnamen aan. Als jullie bestandsnamen anders zijn, kun je in `RUN_FILES` ieder pad afzonderlijk invullen.

`CURRENT_STEP_START_INDEX_COMMON` bevat de 0-based index waarop ieder stroomblok begint. De eerste startindex is altijd 0. Als de blokgrenzen per run verschillen, vul dan `CURRENT_STEP_START_INDEX_BY_RUN` in voor die runs; deze instellingen overschrijven de gemeenschappelijke lijst.

Bijvoorbeeld, `[0, 72, 143, ...]` is alleen een voorbeeldvorm, **geen echte meetgrens**. Gebruik de werkelijke waarden uit jullie meetprogramma/logboek.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- BESTANDEN (PAS AAN AAN JULLIE ECHTE BESTANDSNAMEN) ----------
RUN_NUMBERS = list(range(1, 11))  # runs 1 t/m 10
DMM_FILE_TEMPLATE = "DMMdata_W3_{run}.csv"
SMU_FILE_TEMPLATE = "SMUdata_W3_{run}.csv"  # VUL JULLIE ECHTE SMU-NAAM/TEMPLATE IN

# Alternatief: zet RUN_FILES op een lijst met exact de eigen paden.
# Als RUN_FILES niet None is, wordt deze lijst gebruikt.
RUN_FILES = None
# Voorbeeld van handmatig invullen:
# RUN_FILES = [
#   {"run": 1, "dmm": "DMMdata_pt1000_1.csv", "smu": "SMU_bestandsnaam_run1.csv"},
#   {"run": 2, "dmm": "DMMdata_pt1000_2.csv", "smu": "SMU_bestandsnaam_run2.csv"},
#   ... vul alle 10 runs in ...
# ]

# De CSV's die we gezien hebben: 5 regels metadata, dan één tab-gescheiden numerieke kolom.
DMM_SKIPROWS = 5
SMU_SKIPROWS = 7
DMM_VALUE_COL = 0  # PT1000 weerstand in ohm
SMU_VALUE_COL = 0  # spanning
VOLTAGE_TO_VOLT = 1.0  # 1 voor V, 1e-3 voor mV, 1e-6 voor µV

# ---------- PT1000-KALIBRATIE ----------
# Vul de gemeten weerstanden (in ohm) in bij 77 K en 293 K:
R_77 = 197.5   # VUL IN: Weerstand bij vloeibare stikstof (77 K)
R_293 = 1078.0  # VUL IN: Weerstand bij kamertemperatuur (293 K)

# De constanten a en b worden nu automatisch berekend (T = a * R + b)
PT1000_a = (293 - 77) / (R_293 - R_77)
PT1000_b = 77 - (PT1000_a * R_77)

# ---------- STROOMBLOKKEN ----------
CURRENT_SEQUENCE_A = [0.1, -0.1, 0.3, -0.3, 0.5, -0.5, 0.7, -0.7]
CURRENT_STEP_START_INDEX_COMMON = [0] * len(CURRENT_SEQUENCE_A) # VUL 8 echte startindexen in, eerste is 0
# Optioneel: specifieke blokgrenzen per run, bijv. {1: [0, ...], 2: [0, ...]}
CURRENT_STEP_START_INDEX_BY_RUN = {}

# ---------- ANALYSE ----------
TEMPERATURE_BIN_WIDTH_K = 0.25  # temperatuur-breedte voor het matchen van +I en -I
MIN_POINTS_PER_RUN_BIN = 1       # minimum metingen per stroom/temperatuur-bin binnen een run
TC_RESISTANCE_FRACTION = 0.50   # Tc op 50% van de geschatte normale-toestand-weerstand

# ---------- B-ZELFVELD ----------
# ---------- B-ZELFVELD ----------
SAMPLE_WIDTH_M = 0.003      # VUL IN: breedte w in meter (bijv. 3 mm = 0.003)
SAMPLE_THICKNESS_M = 0.002  # VUL IN: dikte t in meter (bijv. 1 mm = 0.001)

# Effectieve straal op basis van de omtrek: 2*pi*r_eff = 2*(w + t)
SAMPLE_RADIUS_M = (SAMPLE_WIDTH_M + SAMPLE_THICKNESS_M) / np.pi

MU0 = 4 * np.pi * 1e-7



## 2. Bestanden inlezen

De inleesfunctie gebruikt tab als scheidingsteken, slaat de vijf metadataregels over en leest de numerieke kolom. Controleer de gevonden bestandsnamen en meetpunt-aantallen.

In [ ]:
def read_numeric_export(path, skiprows=5, value_col=0):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Bestand niet gevonden: {path}. Controleer de bestandsnaam en zet het CSV-bestand "
            "in dezelfde map als deze notebook, of geef een volledig pad op."
        )
    raw = pd.read_csv(path, sep="\t", header=None, skiprows=skiprows, engine="python")
    if value_col >= raw.shape[1]:
        raise ValueError(f"Kolom {value_col} bestaat niet in {path.name}; gevonden {raw.shape[1]} kolom(men).")
    values = pd.to_numeric(raw.iloc[:, value_col], errors="coerce").dropna().reset_index(drop=True)
    if values.empty:
        raise ValueError(f"Geen numerieke meetwaarden gevonden in {path.name}; controleer SKIPROWS/VALUE_COL.")
    return values.astype(float)

if RUN_FILES is None:
    run_files = [
        {"run": r, "dmm": DMM_FILE_TEMPLATE.format(run=r), "smu": SMU_FILE_TEMPLATE.format(run=r)}
        for r in RUN_NUMBERS
    ]
else:
    run_files = RUN_FILES

print("Aantal geconfigureerde runs:", len(run_files))
for item in run_files:
    print(item)


## 3. Runs koppelen en temperatuur berekenen

De DMM- en SMU-data worden per run gekoppeld op meetindex. Als de aantallen verschillen, gebruikt de code alleen het gemeenschappelijke deel en geeft ze een waarschuwing. Controleer of verschil in aantal meetpunten niet op een verkeerde start/cadans wijst.

In [ ]:
if not (np.isfinite(PT1000_a) and np.isfinite(PT1000_b)):
    raise ValueError("Vul eerst PT1000_a en PT1000_b in met jullie kalibratie.")

run_frames = []
run_length_table = []
for item in run_files:
    run_id = item["run"]
    dmm_R = read_numeric_export(item["dmm"], DMM_SKIPROWS, DMM_VALUE_COL)
    smu_V = read_numeric_export(item["smu"], SMU_SKIPROWS, SMU_VALUE_COL) * VOLTAGE_TO_VOLT
    n = min(len(dmm_R), len(smu_V))
    if len(dmm_R) != len(smu_V):
        print(f"WAARSCHUWING run {run_id}: DMM={len(dmm_R)} punten, SMU={len(smu_V)} punten. "
              f"Alleen eerste {n} punten worden gekoppeld.")
    run_length_table.append({"run": run_id, "n_DMM": len(dmm_R), "n_SMU": len(smu_V), "n_used": n})
    frame = pd.DataFrame({
        "run_id": run_id,
        "sample_index": np.arange(n),
        "R_pt1000_ohm": dmm_R.iloc[:n].to_numpy(),
        "V_measured_V": smu_V.iloc[:n].to_numpy(),
    })
    frame["temperature_K"] = PT1000_a * frame["R_pt1000_ohm"] + PT1000_b
    frame["temperature_C"] = frame["temperature_K"] - 273.15
    run_frames.append(frame)

length_table = pd.DataFrame(run_length_table)
display(length_table)
data_all = pd.concat(run_frames, ignore_index=True)
print("Totaal gekoppelde meetpunten:", len(data_all))
display(data_all.head())


## 4. Iedere run opdelen in de acht stroomblokken

Vul de echte startindexen van alle stroomstappen in. Als dezelfde meetinstellingen en stapduur in alle runs zijn gebruikt, kun je één gemeenschappelijke lijst gebruiken. Als de overgangen per run verschoven zijn, vul dan de indexen per run in `CURRENT_STEP_START_INDEX_BY_RUN` in.

In [ ]:
def get_start_indices(run_id):
    if run_id in CURRENT_STEP_START_INDEX_BY_RUN:
        starts = CURRENT_STEP_START_INDEX_BY_RUN[run_id]
    else:
        starts = CURRENT_STEP_START_INDEX_COMMON
    if len(starts) != len(CURRENT_SEQUENCE_A):
        raise ValueError(
            f"Run {run_id}: vul precies {len(CURRENT_SEQUENCE_A)} stroomblok-startindexen in. "
            "Gebruik CURRENT_STEP_START_INDEX_COMMON of CURRENT_STEP_START_INDEX_BY_RUN."
        )
    starts = np.asarray(starts, dtype=int)
    if starts[0] != 0 or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise ValueError(f"Run {run_id}: startindexen moeten oplopend, uniek en niet-negatief zijn; eerste index is 0.")
    return starts

all_blocks = []
block_summaries = []
for run_id, run_df in data_all.groupby("run_id", sort=True):
    run_df = run_df.sort_values("sample_index").copy()
    starts = get_start_indices(run_id)
    if starts[-1] >= len(run_df):
        raise ValueError(f"Run {run_id}: laatste startindex ({starts[-1]}) ligt buiten de data ({len(run_df)} punten).")
    bounds = list(starts) + [len(run_df)]
    run_df["step_id"] = -1
    run_df["current_A"] = np.nan
    for j, current in enumerate(CURRENT_SEQUENCE_A):
        lo, hi = bounds[j], bounds[j + 1]
        if hi <= lo:
            raise ValueError(f"Run {run_id}, stroomblok {j}: geen meetpunten; controleer de indexen.")
        mask = (run_df["sample_index"] >= lo) & (run_df["sample_index"] < hi)
        run_df.loc[mask, "step_id"] = j
        run_df.loc[mask, "current_A"] = current
    all_blocks.append(run_df)
    block_summaries.append(run_df.groupby(["step_id", "current_A"], as_index=False).agg(
        first_index=("sample_index", "min"), last_index=("sample_index", "max"),
        n_points=("sample_index", "size"), T_min_K=("temperature_K", "min"), T_max_K=("temperature_K", "max")
    ).assign(run_id=run_id))

data = pd.concat(all_blocks, ignore_index=True)
block_summary = pd.concat(block_summaries, ignore_index=True)
display(block_summary[["run_id", "step_id", "current_A", "first_index", "last_index", "n_points", "T_min_K", "T_max_K"]])


## 5. Controleplots van de tien runs

Controleer of de temperatuur tijdens iedere run oploopt en of stroomblokgrenzen goed zijn ingesteld. De stroomplot is een **aangenomen stroomprogramma** op basis van de ingestelde indexen, geen stroommeting uit het bestand.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for run_id, g in data.groupby("run_id"):
    ax.plot(g["sample_index"], g["temperature_K"], label=f"run {run_id}", alpha=0.8)
ax.set_xlabel("Meetindex binnen run")
ax.set_ylabel("Temperatuur (K)")
ax.set_title("PT1000-temperatuur tijdens de runs")
ax.grid(True)
ax.legend(ncol=2)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
example_run = data["run_id"].min()
g = data[data["run_id"] == example_run]
ax.step(g["sample_index"], g["current_A"], where="post")
ax.set_xlabel("Meetindex")
ax.set_ylabel("Ingestelde stroom (A)")
ax.set_title(f"Ingestelde stroomblokken — controle run {example_run}")
ax.grid(True)
plt.show()


## 6. Offsetcompensatie bij vergelijkbare temperatuur

We maken eerst per run, stroomblok en temperatuur-bin een mediane spanning. Daarna worden `+I` en `−I` **binnen dezelfde run en dezelfde temperatuur-bin** aan elkaar gekoppeld.

- Alleen overlappende temperatuur-bins worden gebruikt; er vindt geen extrapolatie plaats.
- Per gepaard punt geldt \(R=(V_+-V_-)/(2|I|)\).
- Als er weinig overlap is, kunnen sommige stroomgroottes geen betrouwbare offsetgecorrigeerde curve opleveren. Dat is een meetbeperking die je moet rapporteren, geen reden om niet-bestaande punten te verzinnen.

Kies `TEMPERATURE_BIN_WIDTH_K` klein genoeg om de transitie te zien, maar groot genoeg dat er metingen in de bins vallen. Controleer achteraf de aantallen per bin.

In [ ]:
if TEMPERATURE_BIN_WIDTH_K <= 0:
    raise ValueError("TEMPERATURE_BIN_WIDTH_K moet positief zijn.")

# Temperatuur-bins; de bin-centra worden gebruikt voor koppelen en plotten.
data = data.copy()
data["T_bin_K"] = (np.floor(data["temperature_K"] / TEMPERATURE_BIN_WIDTH_K) * TEMPERATURE_BIN_WIDTH_K
                   + TEMPERATURE_BIN_WIDTH_K / 2)

per_run_bin = data.groupby(["run_id", "current_A", "T_bin_K"], as_index=False).agg(
    V_median_V=("V_measured_V", "median"),
    T_median_K=("temperature_K", "median"),
    n_points=("V_measured_V", "size")
)
per_run_bin = per_run_bin[per_run_bin["n_points"] >= MIN_POINTS_PER_RUN_BIN].copy()

offset_rows = []
for run_id in sorted(per_run_bin["run_id"].unique()):
    this_run = per_run_bin[per_run_bin["run_id"] == run_id]
    for current_abs in sorted({abs(x) for x in CURRENT_SEQUENCE_A if x > 0}):
        plus = this_run[np.isclose(this_run["current_A"], current_abs)].copy()
        minus = this_run[np.isclose(this_run["current_A"], -current_abs)].copy()
        if plus.empty or minus.empty:
            continue
        matched = plus.merge(minus, on="T_bin_K", suffixes=("_plus", "_minus"), how="inner")
        if matched.empty:
            continue
        matched["run_id"] = run_id
        matched["current_abs_A"] = current_abs
        matched["V_sample_V"] = (matched["V_median_V_plus"] - matched["V_median_V_minus"]) / 2
        matched["R_sample_ohm"] = matched["V_sample_V"] / current_abs
        matched["temperature_K"] = (matched["T_median_K_plus"] + matched["T_median_K_minus"]) / 2
        matched["temperature_C"] = matched["temperature_K"] - 273.15
        matched["delta_T_K"] = abs(matched["T_median_K_plus"] - matched["T_median_K_minus"])
        matched["n_plus"] = matched["n_points_plus"]
        matched["n_minus"] = matched["n_points_minus"]
        offset_rows.append(matched[["run_id", "current_abs_A", "T_bin_K", "temperature_K", "temperature_C",
                                    "V_median_V_plus", "V_median_V_minus", "V_sample_V", "R_sample_ohm",
                                    "delta_T_K", "n_plus", "n_minus"]])

if offset_rows:
    paired = pd.concat(offset_rows, ignore_index=True).sort_values(["current_abs_A", "run_id", "temperature_K"])
else:
    paired = pd.DataFrame(columns=["run_id", "current_abs_A", "T_bin_K", "temperature_K", "temperature_C",
                                   "V_median_V_plus", "V_median_V_minus", "V_sample_V", "R_sample_ohm",
                                   "delta_T_K", "n_plus", "n_minus"])

if paired.empty:
    print("Geen bij dezelfde temperatuur gekoppelde +I/-I-punten gevonden.")
    print("Controleer stroomblok-indexen, kalibratie en overlap van de temperatuurgebieden."
          " Als de blokken geen temperatuuroverlap hebben, kan offsetcompensatie niet betrouwbaar uit deze reeks worden berekend.")
else:
    print("Aantal offsetgecorrigeerde punten:", len(paired))
    overlap_summary = paired.groupby("current_abs_A").agg(
        n_pairs=("R_sample_ohm", "size"),
        n_runs=("run_id", "nunique"),
        T_min_K=("temperature_K", "min"),
        T_max_K=("temperature_K", "max"),
        mean_delta_T_K=("delta_T_K", "mean"),
        max_delta_T_K=("delta_T_K", "max")
    ).reset_index()
    display(overlap_summary)
    display(paired.head())


## 7. Weerstandscurves en spreiding over runs

De grafiek toont per stroomgrootte de gemiddelde weerstand per temperatuur-bin. De foutbalken tonen één standaarddeviatie tussen de beschikbare runs op die temperatuur-bin; ze zijn niet automatisch de volledige meetonzekerheid.

In [ ]:
if paired.empty:
    print("Geen R(T)-plot mogelijk totdat er gepaarde temperatuur-bins zijn.")
else:
    curve_summary = paired.groupby(["current_abs_A", "T_bin_K"], as_index=False).agg(
        temperature_K=("temperature_K", "mean"),
        R_mean_ohm=("R_sample_ohm", "mean"),
        R_median_ohm=("R_sample_ohm", "median"),
        R_std_ohm=("R_sample_ohm", "std"),
        n_runs=("run_id", "nunique"),
        mean_delta_T_K=("delta_T_K", "mean")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    for current, g in curve_summary.groupby("current_abs_A"):
        g = g.sort_values("temperature_K")
        yerr = g["R_std_ohm"].fillna(0)
        ax.errorbar(g["temperature_K"], g["R_mean_ohm"], yerr=yerr, marker="o", linestyle="-",
                    capsize=2, label=f"|I| = {current:g} A")
    ax.set_xlabel("Temperatuur (K)")
    ax.set_ylabel("Offsetgecorrigeerde weerstand (Ω)")
    ax.set_title("R(T) per stroomgrootte; punten alleen waar +I/−I overlappen")
    ax.grid(True)
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    for current, g in curve_summary.groupby("current_abs_A"):
        ax.plot(g["temperature_K"], g["n_runs"], "o-", label=f"|I| = {current:g} A")
    ax.set_xlabel("Temperatuur (K)")
    ax.set_ylabel("Aantal runs in dat temperatuur-bin")
    ax.set_title("Datadekking per temperatuur")
    ax.grid(True)
    ax.legend()
    plt.show()


## 8. Schatting van Tc en vergelijking tussen de tien runs

We definiëren hier \(T_c\) als de temperatuur waarop de weerstand een ingestelde fractie van de normale-toestand-weerstand bereikt (standaard 50%). Dit is een operationele definitie; noem die expliciet in het verslag.

De notebook schat zowel een \(T_c\) uit de gemiddelde curve als een \(T_c\) per individuele run, als die run voldoende gepaarde punten en een duidelijke drempelkruising heeft. De spreiding tussen runs is een maat voor reproduceerbaarheid, niet de enige bron van meetonzekerheid.

In [ ]:
def estimate_tc(group, fraction=0.50):
    g = group.sort_values("temperature_K").dropna(subset=["temperature_K", "R_sample_ohm"]).copy()
    if len(g) < 3:
        return np.nan
    n_high = max(2, int(np.ceil(0.2 * len(g))))
    R_normal = g.tail(n_high)["R_sample_ohm"].median()
    if not np.isfinite(R_normal) or R_normal <= 0:
        return np.nan
    threshold = fraction * R_normal
    x = g["temperature_K"].to_numpy()
    y = g["R_sample_ohm"].to_numpy()
    for i in range(1, len(y)):
        # Zoek de overgang van lage weerstand naar de drempel bij oplopende temperatuur.
        if y[i-1] <= threshold <= y[i]:
            if y[i] == y[i-1]:
                return float((x[i-1] + x[i]) / 2)
            return float(x[i-1] + (threshold-y[i-1])*(x[i]-x[i-1])/(y[i]-y[i-1]))
    return np.nan

if paired.empty:
    tc_per_run = pd.DataFrame()
    tc_summary = pd.DataFrame()
    tc_curve_table = pd.DataFrame()
else:
    tc_rows = []
    for (run_id, current), g in paired.groupby(["run_id", "current_abs_A"]):
        tc_k = estimate_tc(g, TC_RESISTANCE_FRACTION)
        tc_rows.append({"run_id": run_id, "current_A": current, "Tc_K": tc_k,
                        "Tc_C": tc_k - 273.15 if np.isfinite(tc_k) else np.nan})
    tc_per_run = pd.DataFrame(tc_rows)
    tc_summary = tc_per_run.groupby("current_A", as_index=False).agg(
        n_valid_runs=("Tc_K", "count"), Tc_mean_K=("Tc_K", "mean"), Tc_std_K=("Tc_K", "std"),
        Tc_mean_C=("Tc_C", "mean"), Tc_std_C=("Tc_C", "std")
    )
    display(tc_per_run)
    display(tc_summary)

    tc_curve_rows = []
    for current, g in paired.groupby("current_abs_A"):
        curve = g.groupby("T_bin_K", as_index=False).agg(
            temperature_K=("temperature_K", "mean"),
            R_sample_ohm=("R_sample_ohm", "mean"),
            n_runs=("run_id", "nunique")
        )
        tc_k = estimate_tc(curve, TC_RESISTANCE_FRACTION)
        tc_curve_rows.append({"current_A": current, "Tc_from_mean_curve_K": tc_k,
                              "Tc_from_mean_curve_C": tc_k - 273.15 if np.isfinite(tc_k) else np.nan,
                              "n_temperature_bins": len(curve)})
    tc_curve_table = pd.DataFrame(tc_curve_rows)
    display(tc_curve_table)


## 9. Verkennende B–T-plot

Als eerste geometrische benadering gebruiken we het zelfveld aan het oppervlak van een lange rechte draad:
\[
B=\frac{\mu_0 I}{2\pi r}.
\]
Hier is \(r\) de straal van het **supergeleidermateriaal** in meter. Dit is een geschat zelfveld; het is niet hetzelfde als een direct gemeten extern kritisch veld. De interpretatie van een B–T-plot hangt af van de geometrie van het sample en het soort supergeleider.

In [ ]:
if not np.isfinite(SAMPLE_RADIUS_M) or SAMPLE_RADIUS_M <= 0:
    print("Vul SAMPLE_RADIUS_M in om B te berekenen.")
elif tc_summary.empty:
    print("Geen Tc-samenvatting beschikbaar.")
else:
    bt = tc_summary.dropna(subset=["Tc_mean_K"]).copy()
    bt["B_self_T"] = MU0 * bt["current_A"] / (2 * np.pi * SAMPLE_RADIUS_M)
    display(bt[["current_A", "B_self_T", "Tc_mean_K", "Tc_std_K", "n_valid_runs"]])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(bt["Tc_mean_K"], bt["B_self_T"], xerr=bt["Tc_std_K"], marker="o", linestyle="none", capsize=3)
    for _, row in bt.iterrows():
        ax.annotate(f'{row["current_A"]:g} A', (row["Tc_mean_K"], row["B_self_T"]),
                    xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel("Gemiddelde Tc (K)")
    ax.set_ylabel("Geschat zelfveld B (T)")
    ax.set_title("Verkennende B–T-plot (horizontale foutbalk = spreiding Tc tussen runs)")
    ax.grid(True)
    plt.show()


## 10. Resultaten opslaan

CSV-uitvoer bevat de gekoppelde ruwe data, stroomblokindeling, offsetgecorrigeerde meetpunten en Tc-tabellen. Gebruik de bestanden pas in het verslag nadat de blokgrenzen, kalibratie en temperatuur-overlap gecontroleerd zijn.

In [ ]:
data.to_csv("ten_runs_combined_DMM_SMU.csv", index=False)
block_summary.to_csv("ten_runs_current_blocks.csv", index=False)
per_run_bin.to_csv("ten_runs_voltage_temperature_bins.csv", index=False)
if not paired.empty:
    paired.to_csv("ten_runs_offset_corrected.csv", index=False)
if not tc_per_run.empty:
    tc_per_run.to_csv("ten_runs_Tc_per_run.csv", index=False)
if not tc_summary.empty:
    tc_summary.to_csv("ten_runs_Tc_summary.csv", index=False)
if not tc_curve_table.empty:
    tc_curve_table.to_csv("ten_runs_Tc_from_mean_curves.csv", index=False)
print("CSV-export klaar.")
